# env_gen

> Generate an environment.yml from an nbdev-style settings.ini

In [ ]:
#| default_exp env_gen

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import argparse
import configparser
import os
import shlex
import sys
from typing import List, Tuple

In [ ]:
#| export
DEFAULT_ENV_NAME = "nbdev-env"
DEFAULT_CHANNELS = ["conda-forge"]

## Reading Settings

In [ ]:
#| export
def read_settings(
    path: str  # Path to settings.ini file
) -> configparser.ConfigParser:  # Configured ConfigParser object
    "Read settings.ini file with percent-style interpolation."
    cp = configparser.ConfigParser(
        interpolation=configparser.BasicInterpolation(),
        delimiters=("=",),
    )
    if not os.path.exists(path):
        sys.exit(f"ERROR: settings.ini not found at: {path}")
    with open(path, "r", encoding="utf-8") as f:
        cp.read_file(f)
    return cp

In [ ]:
#| export
def get_section(
    cp: configparser.ConfigParser  # ConfigParser object
): 
    "Get configuration section from ConfigParser. nbdev puts keys at top-level (DEFAULT). Falls back to first section if present."
    if cp.defaults():
        return cp["DEFAULT"]
    elif cp.sections():
        return cp[cp.sections()[0]]
    else:
        return {}

## Utility Functions

In [ ]:
#| export
def split_list(
    val: str | None  # String value to split (can be None)
) -> List[str]:  # List of unique, non-empty strings preserving order
    "Split space- and/or comma-separated lists while respecting quotes."
    if not val: 
        return []
    # Normalize commas to spaces; shlex handles quotes nicely
    norm = val.replace(",", " ")
    items = shlex.split(norm)
    # Strip and drop empties
    items = [x.strip() for x in items if x.strip()]
    # De-duplicate preserving order
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

In [ ]:
#| export
def normalize_python_spec(
    min_py: str=None  # Minimum Python version string
) -> str:  # Normalized Python specification string or None
    """Normalize Python version specification for conda. Accepts ">=3.10", "3.10", "3.9.*", etc. If bare like "3.10", makes it ">=3.10"."""
    if not min_py: 
        return None
    s = min_py.strip()
    # If bare version like "3.10", make it ">=3.10"
    if s[0].isdigit():
        s = f">={s}"
    # conda accepts comparison operators in specs
    return f"python{s}"

## Configuration Collection

In [ ]:
#| export
def collect_values(
    cfg    # Configuration section dictionary - TODO: Add type hint
) -> Tuple[str, List[str], List[str], List[str], List[str]]:  # Tuple of (env_name, channels, conda_reqs, pip_reqs, dev_pip_reqs)
    "Collect configuration values from settings."
    lib_name = cfg.get("lib_name", "").strip()
    env_name = f"{lib_name}-dev" if lib_name else DEFAULT_ENV_NAME

    channels = split_list(cfg.get("conda_channels"))
    if not channels:
        channels = DEFAULT_CHANNELS[:]  # default if none provided

    conda_reqs = split_list(cfg.get("conda_requirements"))
    pip_reqs = split_list(cfg.get("requirements"))
    dev_pip_reqs = split_list(cfg.get("dev_requirements"))

    python_spec = normalize_python_spec(cfg.get("min_python"))
    if python_spec:
        # Put python at the front
        conda_reqs = [python_spec] + [x for x in conda_reqs if not x.lower().startswith("python")]
    return env_name, channels, conda_reqs, pip_reqs, dev_pip_reqs

## YAML Generation

In [ ]:
#| export
def emit_yaml(
    name: str,    # Environment name
    channels: List[str],    # List of conda channels
    conda_deps: List[str],    # List of conda dependencies
    pip_lines: List[str],    # List of pip packages
) -> str:  # YAML formatted string
    "Generate environment.yml content. Manual YAML emitter to avoid extra dependencies."
    lines = []
    lines.append(f"name: {name}")
    lines.append("channels:")
    for ch in channels:
        lines.append(f"  - {ch}")
    lines.append("dependencies:")
    for dep in conda_deps:
        lines.append(f"  - {dep}")
    if pip_lines:
        # Ensure pip itself is installed
        if not any(d.strip().lower() == "pip" for d in conda_deps):
            lines.append("  - pip")
        lines.append("  - pip:")
        for pkg in pip_lines:
            lines.append(f"      - {pkg}")
    return "\n".join(lines) + "\n"

## CLI Interface

In [ ]:
#| export
def main():
    """Generate environment.yml from nbdev settings.ini.
    
    Reads: lib_name, min_python, conda_channels, conda_requirements,
           requirements, dev_requirements (all optional).
    Writes: environment.yml (or stdout).
    Adds an editable pip install (-e ".[EXTRAS]") by default.
    
    Examples:
        python gen_env_from_settings.py
        python gen_env_from_settings.py --name myproj-dev --extras dev,docs --out env.yml
        python gen_env_from_settings.py --no-editable --stdout
        python gen_env_from_settings.py --no-include-pip-reqs
    """
    ap = argparse.ArgumentParser(description="Generate environment.yml from nbdev settings.ini")
    ap.add_argument("--settings", default="settings.ini", help="Path to settings.ini")
    ap.add_argument("--name", default=None, help="Environment name override")
    ap.add_argument("--extras", default="dev",
                    help="Editable pip extras to install, e.g. 'dev' or 'dev,docs'. Use '' to install no extras.")
    ap.add_argument("--no-editable", action="store_true",
                    help="Do not include editable install (-e .).")
    ap.add_argument("--include-pip-reqs", dest="include_pip_reqs", action="store_true",
                    help="Also include 'requirements' and 'dev_requirements' from settings.ini in pip section.")
    ap.add_argument("--no-include-pip-reqs", dest="include_pip_reqs", action="store_false",
                    help="Do not include pip requirements from settings.ini (default).")
    ap.set_defaults(include_pip_reqs=False)
    ap.add_argument("--out", default="environment.yml", help="Output file path.")
    ap.add_argument("--stdout", action="store_true", help="Write YAML to stdout instead of a file.")
    args = ap.parse_args()

    cp = read_settings(args.settings)
    cfg = get_section(cp)
    env_name, channels, conda_reqs, pip_reqs, dev_pip_reqs = collect_values(cfg)

    # Build pip section
    pip_lines: List[str] = []
    if not args.no_editable:
        extras = args.extras.strip()
        if extras:
            # Normalize "dev,docs" -> .[dev,docs]
            extras_clean = ",".join([x.strip() for x in extras.split(",") if x.strip()])
            pip_lines.append(f'-e ".[{extras_clean}]"')
        else:
            pip_lines.append("-e .")
    if args.include_pip_reqs:
        pip_lines.extend(pip_reqs)
        pip_lines.extend(dev_pip_reqs)

    # Emit YAML
    name = args.name or env_name
    yaml_text = emit_yaml(name=name, channels=channels, conda_deps=conda_reqs, pip_lines=pip_lines)

    if args.stdout:
        sys.stdout.write(yaml_text)
    else:
        with open(args.out, "w", encoding="utf-8") as f:
            f.write(yaml_text)
        print(f"Wrote {args.out}")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()